In [85]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz, load_npz
from sklearn.metrics.pairwise import cosine_similarity
import os
import kagglehub

In [86]:
DATA_DIR = os.path.join("..", "data")  # same relative-path pattern as before

train = pd.read_parquet(os.path.join(DATA_DIR, "train_ratings.parquet"))
test = pd.read_parquet(os.path.join(DATA_DIR, "test_ratings.parquet"))

train_sample = train.sample(5_000_000, random_state=42) #Allows for cells to run without taking forever
test_sample = test.sample(1_000_000, random_state=42)

print(train_sample.shape, test_sample.shape)

(5000000, 4) (1000000, 4)


## Baseline 1 (K most popular shows)

In [87]:
# Count positive interactions per anime, using train only
popularity = train_sample[train_sample['is_positive'] == 1].groupby('anime_id').size().sort_values(ascending=False)

#K most popular shows
K = 10
top_k_popular = popularity.head(K).index.tolist()

print(top_k_popular)

#Find the precision and recall rate
def precision_recall_at_k(test_df, recommended_items, k):
    total_relevant = 0
    total_recommended_relevant = 0
    
    for user_id, group in test_df[test_df['is_positive'] == 1].groupby('user_id'):
        actual_positive = set(group['anime_id'])
        recommended = set(recommended_items[:k])
        
        hits_this_user = len(actual_positive & recommended)
        total_recommended_relevant += hits_this_user
        total_relevant += len(actual_positive)
    
    # Recall: of everything the user actually liked, what fraction did we recommend?
    recall = total_recommended_relevant / total_relevant if total_relevant > 0 else 0
    
    # Precision: of everything we recommended, what fraction did users actually like?
    num_users = test_df['user_id'].nunique()
    precision = total_recommended_relevant / (num_users * k)
    
    return precision, recall

precision, recall = precision_recall_at_k(test_sample, top_k_popular, K)
print(f"Recall@{K}: {recall:.4f}")
print(f"Precision@{K}: {precision:.4f}")


[20, 2, 92, 2376, 99, 726, 1147, 100, 1160, 1167]
Recall@10: 0.0669
Precision@10: 0.0072


## Baseline 2 (Item-Item Collabritive Filtering)

In [88]:
anime_ids = train_sample['anime_id'].astype('category')
anime_id_map = dict(enumerate(anime_ids.cat.categories))      
anime_id_map_reverse = {v: k for k, v in anime_id_map.items()} 

user_ids = train_sample['user_id'].astype('category')
user_id_map = dict(enumerate(user_ids.cat.categories))
user_id_map_reverse = {v: k for k, v in user_id_map.items()}

train_sample['anime_idx'] = anime_ids.cat.codes
train_sample['user_idx'] = user_ids.cat.codes

test_sample['anime_idx'] = test_sample['anime_id'].map(anime_id_map_reverse)
test_sample['user_idx'] = test_sample['user_id'].map(user_id_map_reverse)


p_train = train_sample[train_sample['is_positive']==1]
test_sample_clean = test_sample.dropna(subset=['anime_idx', 'user_idx'])

In [90]:
item_user_matrix = csr_matrix(
    (
    [1] * len(p_train), (p_train['anime_idx'], p_train['user_idx'])
    ),
    shape = (len(anime_id_map), len(user_id_map))
)

item_similarity = cosine_similarity(item_user_matrix, dense_output=False)
save_npz(os.path.join(DATA_DIR, "item_user_matrix.npz"), item_user_matrix)
save_npz(os.path.join(DATA_DIR, "item_similarity.npz"), item_similarity)

In [92]:
def recommend_for_user(user_idx, k=10):
    # Get this user's positively-rated anime (as matrix indices)
    user_rated = p_train[p_train['user_idx'] == user_idx]['anime_idx'].tolist()
    
    if not user_rated:
        return []  
    

    scores = np.asarray(item_similarity[user_rated].sum(axis=0)).flatten()
    
    scores[user_rated] = -1
    
    top_idx = scores.argsort()[::-1][:k]
    
    return [anime_id_map[i] for i in top_idx]

sample_user_idx = 5
recs = recommend_for_user(sample_user_idx, k=10)

path = kagglehub.dataset_download("ramazanturann/user-animelist-dataset")
animes = pd.read_csv(os.path.join(path, "animes.csv"))

print(f'10 Recommend animes: \n\n {animes[animes['animeID'].isin(recs)][['animeID', 'title']]}')

10 Recommend animes: 

        animeID                               title
2844      2845                          Cyborg 009
4079      4080                   Here is Greenwood
7967      7968                   Rokudenashi Blues
9891      9892  Omishi Mahou Gekijou: Risky★Safety
10171    10172             One: Kagayaku Kisetsu e
11126    11127                  Prism Ark Specials
13928    13929                  Ginsoukikou Ordian
14419    14420       Dinosaur Expedition Born Free
14655    14656      Negadon: The Monster from Mars
15506    15507                   Boku no Son Gokuu


In [94]:
def precision_recall_itemcf(test_df, k=10, sample_users=None):
    total_relevant = 0
    total_recommended_relevant = 0
    
    test_positive = test_df[test_df['is_positive'] == 1]
    grouped = test_positive.groupby('user_idx')
    
    sample_uid = list(grouped.groups.keys())[0]

    print("Recommended:", recommend_for_user(sample_uid, k=10))
    print("Actually liked (test):", grouped.get_group(sample_uid)['anime_id'].tolist())
    
    users_to_eval = list(grouped.groups.keys())
    if sample_users:
        users_to_eval = np.random.choice(users_to_eval, size=sample_users, replace=False)
    
    for user_idx in users_to_eval:
        group = grouped.get_group(user_idx)
        actual_positive = set(group['anime_id'])
        
        recommended = set(recommend_for_user(user_idx, k=k))  # ← computed fresh, per user
        
        hits_this_user = len(actual_positive & recommended)
        total_recommended_relevant += hits_this_user
        total_relevant += len(actual_positive)
    
    recall = total_recommended_relevant / total_relevant if total_relevant > 0 else 0
    precision = total_recommended_relevant / (len(users_to_eval) * k)
    
    return precision, recall

precision_recall_itemcf(test_sample_clean, k=10, sample_users=2000)

Recommended: [15507, 7968, 14656, 4080, 2845, 10172, 11127, 13929, 14420, 9892]
Actually liked (test): [283, 631]


(0.0051, 0.03230915426037377)